# Connect resources — CATIA 3DEXPERIENCE

Register a **connected** CATIA assembly from **3DEXPERIENCE** without uploading geometry, then run **`@istari:extract`** on an agent with the CATIA tool.

The pointer file is JSON metadata (`product_id`, revisions) uploaded with extension `.istari_dassault_3dexperience_catia_metadata`. The CAD model stays in 3DEXPERIENCE; Istari stores only the link.

You will:

1. Connect to the Istari Digital Platform with a personal access token
2. Register the 3DEXPERIENCE pointer as a Model resource
3. Submit `@istari:extract` and wait for completion

Uses the official v2 **`istari_digital_client.Client`** (same pattern as [`connect-resources-twc.ipynb`](./connect-resources-twc.ipynb)).

### Prerequisites

- **`istari-digital-client`** and **`python-dotenv`** (from cookbook root: `uv sync --group dev`).
- Access to **`@istari:extract`** with tool **`dassault_3dexperience_catia`** on a Windows agent.
- In [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN` (do not commit `.env`).
- A **product id** (and revisions) from 3DEXPERIENCE — set `PRODUCT_ID` in **Prep**.

### Install kernel (optional)

From the cookbook repository root:

```bash
uv sync --group dev
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Select **Python (istari-client-cookbook)** in the kernel picker.

### Running order

Run **Setup → Connect**, then **Prep**, then §1–§2 in order.

> **Pause in the web app:** after §1, open the new Model under **Files**; after §2, open **Jobs** to watch `@istari:extract` run on the agent.

## Setup

**Connect** loads credentials and constructs `Client`. **Prep** sets the 3DEXPERIENCE pointer fields and CATIA tool metadata — edit before the demo cells.

### Connect

Load [`samples/.env`](../.env), build `Configuration`, and create `Client` — the same pattern as other cookbook resource recipes.

In [ ]:
import os
from pathlib import Path

import dotenv
from istari_digital_client import Client, Configuration

_cwd = Path.cwd()
SAMPLES_DIR = _cwd.parent if (_cwd.parent / ".env").exists() else _cwd / "samples"

dotenv.load_dotenv(SAMPLES_DIR / ".env", override=True)
registry_url = os.environ.get("ISTARI_REGISTRY_URL")
token = os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
if not registry_url or not token:
    raise RuntimeError("Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env")

client = Client(Configuration(registry_url=registry_url, registry_auth_token=token))

me = client.get_current_user()
print(f"Registry: {registry_url}")
print(f"Signed in as: {me.display_name} ({me.email})")

### Prep

3DEXPERIENCE product reference, display names for the connected resource, and CATIA tool settings for the extract job. Change `PRODUCT_ID` for your assembly.

In [ ]:
import json
import re
import tempfile
from pathlib import Path
from time import sleep

from istari_digital_client import JobStatusName

POINTER_SUFFIX = ".istari_dassault_3dexperience_catia_metadata"

# From 3DEXPERIENCE — geometry is not uploaded; only this metadata is registered.
PRODUCT_ID = "prd00000947"
MAJOR_REVISION = "---"
MINOR_REVISION = ""

POINTER_DISPLAY_NAME = "UAV Assembly"
POINTER_EXTERNAL_ID = "uav-catia-3dx-connected"

CATIA_TOOL_NAME = "dassault_3dexperience_catia"
CATIA_TOOL_VERSION = "R2023x"
CATIA_OPERATING_SYSTEM = "Windows 11"

_match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", registry_url)
ui_base = registry_url.rstrip("/") if not _match else f"{_match.group(1)}{_match.group(2)}"


def catia_pointer_payload(product_id: str, *, major: str, minor: str = "") -> dict:
    """3DEXPERIENCE pointer file body for a connected CATIA resource."""
    return {
        "product_id": product_id,
        "major_revision": major,
        "minor_revision": minor,
    }


print(f"Product id: {PRODUCT_ID}")
print(f"CATIA tool: {CATIA_TOOL_NAME} {CATIA_TOOL_VERSION} on {CATIA_OPERATING_SYSTEM}")

## 1 · Register the connected CATIA pointer

Upload JSON pointer metadata via `add_model` (temp file required by the SDK). The assembly remains in 3DEXPERIENCE; Istari registers a Model resource that references it.

In [ ]:
pointer_payload = catia_pointer_payload(
    PRODUCT_ID,
    major=MAJOR_REVISION,
    minor=MINOR_REVISION,
)
print(f"Pointer file content ({POINTER_SUFFIX}):")
print(json.dumps(pointer_payload, indent=2))

_pointer_tmp = tempfile.NamedTemporaryFile(
    mode="w", encoding="utf-8", suffix=POINTER_SUFFIX, delete=False
)
json.dump(pointer_payload, _pointer_tmp, indent=2)
_pointer_tmp.write("\n")
_pointer_tmp.close()
_pointer_path = Path(_pointer_tmp.name)

try:
    print(f"Uploading pointer {_pointer_path.name}...")
    model = client.add_model(
        path=_pointer_path,
        external_identifier=POINTER_EXTERNAL_ID,
        display_name=POINTER_DISPLAY_NAME,
        description="Connected reference to a CATIA assembly in 3DEXPERIENCE",
    )
finally:
    _pointer_path.unlink(missing_ok=True)

print(f"Connected model id: {model.id}")
print(f"UI: {ui_base}/files/{model.id}")

## 2 · Run `@istari:extract`

Submit an extract job against the pointer Model from §1. The agent opens CATIA on the VM and writes neutral artifacts (parameters, mass, views, and similar) back to the resource.

Expect roughly 60–120 seconds depending on model size and agent load.

In [ ]:
job = client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name=CATIA_TOOL_NAME,
    tool_version=CATIA_TOOL_VERSION,
    operating_system=CATIA_OPERATING_SYSTEM,
)
print(f"Job {job.id} submitted")
print(f"UI: {ui_base}/jobs/{job.id}")

### Poll until the job finishes

Poll every five seconds until status is **Completed**, **Failed**, or **Canceled**.

In [ ]:
elapsed = 0
poll_interval = 5

while True:
    job = client.get_job(job.id)
    status = job.status.name
    print(f"{elapsed}s: {status.value}")
    if status in (
        JobStatusName.COMPLETED,
        JobStatusName.FAILED,
        JobStatusName.CANCELED,
    ):
        break
    sleep(poll_interval)
    elapsed += poll_interval

if job.status.name != JobStatusName.COMPLETED:
    msg = job.status.message or ""
    raise RuntimeError(f"Job {job.id} ended with {job.status.name!s}. {msg}")

model = client.get_model(model.id)
print("\nArtifacts on the connected model:")
for artifact in model.artifacts or []:
    print(f"  {artifact.name}")

## Learn more

- [Connect resources — Teamwork Cloud](./connect-resources-twc.ipynb) — same pointer pattern for TWC
- [Using resources](./using-resources.ipynb) — upload, search, and share platform resources
- [Chaining jobs](../chaining_jobs.ipynb) — job submission with `istari_labs_helpers`